# ❤️ MIT-BIH Arrhythmia Dataset — Interactive Preview

Preview the [MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0/) right inside **Google Colab** — no local setup needed.

Part of [CardioEdgeAI](https://github.com/md-rejoyan-islam/CardioEdgeAI); mirrors the local pipeline (`src/data/`): download → filter → R-peak → beat windows.

**What you will see:** raw ECG · filtering effect · R-peak detection · N/S/V beat windows · class imbalance.


In [ ]:
# 1 — install the ECG toolkit (Colab preinstalls numpy/scipy/matplotlib)
%pip install -q wfdb neurokit2

In [ ]:
# 2 — download a small record set (~15 MB, seconds in Colab)
#      100: mostly normal + A-type S | 234: V beats | others: variety
import wfdb, os
RECORDS = ['100', '103', '105', '200', '202', '234']
os.makedirs('mitdb', exist_ok=True)
wfdb.dl_database('mitdb', dl_dir='mitdb', records=RECORDS)
print('downloaded:', sorted(os.listdir('mitdb')))

In [ ]:
# 3 — first 10 s of record 100 (MLII lead @ 360 Hz), RAW
import wfdb, numpy as np, matplotlib.pyplot as plt
rec = wfdb.rdrecord('mitdb/100')
fs = rec.fs
signal = rec.p_signal[:, 0]
n = 10 * fs
t = np.arange(n) / fs
plt.figure(figsize=(12, 3))
plt.plot(t, signal[:n], lw=0.7, color='tab:red')
plt.xlabel('time (s)'); plt.ylabel('mV'); plt.grid(alpha=0.3)
plt.title(f'Record 100 — MLII lead @ {fs} Hz (raw)')
plt.show()

In [ ]:
# 4 — the CardioEdgeAI preprocessing: zero-phase 0.5–40 Hz bandpass
from scipy.signal import butter, sosfiltfilt
sos = butter(4, [0.5, 40], btype='bandpass', fs=fs, output='sos')
filtered = sosfiltfilt(sos, signal)
plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1); plt.plot(t, signal[:n], lw=0.7, color='tab:red')
plt.ylabel('mV (raw)'); plt.grid(alpha=0.3); plt.title('raw vs filtered')
plt.subplot(2, 1, 2); plt.plot(t, filtered[:n], lw=0.7, color='tab:blue')
plt.xlabel('time (s)'); plt.ylabel('mV (filtered)'); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 5 — R-peak detection (same method as the deployed pipeline)
import neurokit2 as nk
_, info = nk.ecg_peaks(filtered, sampling_rate=fs, method='neurokit')
r = info['ECG_R_Peaks']
r_in = r[(r >= 0) & (r < n)]
plt.figure(figsize=(12, 3))
plt.plot(t, filtered[:n], lw=0.7, color='tab:blue')
plt.plot(r_in / fs, filtered[r_in], 'rv', ms=6, label='R peak')
plt.xlabel('time (s)'); plt.ylabel('mV'); plt.legend(); plt.grid(alpha=0.3)
plt.title(f'R-peak detection — {len(r_in)} beats in 10 s (~{len(r_in)*6} bpm)')
plt.show()

In [ ]:
# 6 — beat windows per class: N / S / V (R−0.4 s .. R+0.6 s, like training)
BEFORE, AFTER = int(0.4*fs), int(0.6*fs)
AAMI = {'N': 'NLR.ej', 'S': 'AaJS', 'V': 'VE'}
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, (rec_id, group) in zip(axes, [('100','N'), ('100','S'), ('234','V')]):
    r2 = wfdb.rdrecord(f'mitdb/{rec_id}'); ann = wfdb.rdann(f'mitdb/{rec_id}', 'atr')
    sig = sosfiltfilt(sos, r2.p_signal[:, 0]); sig = (sig - sig.mean()) / (sig.std() + 1e-9)
    idx = next(i for i, (s, sym) in enumerate(zip(ann.sample, ann.symbol))
               if sym in AAMI[group] and ann.sample[i] > BEFORE
               and ann.sample[i] + AFTER < len(sig))
    w = sig[ann.sample[idx]-BEFORE : ann.sample[idx]+AFTER]
    tt = (np.arange(len(w)) - BEFORE) / fs
    ax.plot(tt, w, lw=1.2, color='tab:blue'); ax.axvline(0, ls='--', c='gray', lw=0.8)
    ax.set_title(f'class {group} (record {rec_id})'); ax.set_xlabel('time from R (s)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('z-scored mV')
plt.suptitle('the 360-sample windows the model actually sees', y=1.02)
plt.show()

In [ ]:
# 7 — class imbalance across the downloaded records
from collections import Counter
c = Counter()
for rec_id in RECORDS:
    ann = wfdb.rdann(f'mitdb/{rec_id}', 'atr')
    for sym in ann.symbol:
        for g, syms in AAMI.items():
            if sym in syms: c[g] += 1
plt.figure(figsize=(5, 3))
plt.bar(c.keys(), c.values(), color=['#4CAF50', '#FF9800', '#F44336'])
for g, v in c.items(): plt.text(g, v, str(v), ha='center', va='bottom')
plt.title('beat class counts (downloaded records)')
plt.show()
print('S is the rare class — why training uses class weights (docs/PHASE4.md)')

## 🚀 Going further

This preview mirrors the full CardioEdgeAI pipeline. To reproduce the complete project (dataset build → CNN training → INT8 quantization → ESP32 deployment):

```bash
!git clone https://github.com/md-rejoyan-islam/CardioEdgeAI.git
```

Then follow *Reproducing everything* in [docs/RESULTS.md](https://github.com/md-rejoyan-islam/CardioEdgeAI/blob/main/docs/RESULTS.md). Data: MIT-BIH (PhysioNet) — cite Moody & Mark (2001).